# Creating our own network in pytorch

prerequisites - understand what are DataSets and Data Loaders
understand what is an MLP



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

# Load dataset
data = load_iris()
X = data.data
y = data.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a model
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate
print("Test Accuracy:", model.score(X_test, y_test))

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets list -s "iris"


In [ ]:
!kaggle datasets download -d uciml/iris

In [ ]:
!mkdir datasets

In [ ]:

!unzip -q iris.zip -d datasets

In [ ]:
!ls datasets

In [ ]:
import pandas as pd
iris_data = pd.read_csv('datasets/Iris.csv')

In [ ]:
iris_data.head()

In [ ]:
# Replace the 'Species' column with numerical values
# OLD:
# iris_data['Species'].replace({'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}, inplace=True)

# New
species_mapping = {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}
iris_data['Species'] = iris_data['Species'].map(species_mapping)

In [ ]:
iris_data.head()

In [ ]:
# remove the Id column
iris_data.drop('Id', axis=1, inplace=True)

In [ ]:
X = iris_data.drop('Species', axis=1)
y = iris_data['Species']

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X_np = X.values
y_np = y.values

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")


X_train_torch = torch.tensor(X_train,dtype=torch.float32, device = device) # defined by data type of input - ALWAYS BE SURE THIS IS WHAT YOU WANT
X_test_torch = torch.tensor(X_test, dtype=torch.float32, device = device) # force float32 - SAFER
y_train_torch = torch.tensor(y_train, dtype=torch.float32, device = device)
y_test_torch = torch.tensor(y_test, dtype=torch.float32, device = device)

In [ ]:
X_train_torch[0:10]

In [ ]:
y_train_torch[0:10]

In [ ]:
# Create a Model class, that inherits nn.Module
# simple, Functional API, using the nn.Module syntax

class MyModel(nn.Module):
  def __init__(self, input_features=4, hidden_1 = 8, hidden_2 = 4, output_features = 1):
    super().__init__()
    self.FullyConnected_1 = nn.Linear(input_features, hidden_1)
    self.FullyConnected_2 = nn.Linear(hidden_1, hidden_2)
    self.OutputLayer = nn.Linear(hidden_2, output_features)

  def forward(self, x):
    x = F.relu(self.FullyConnected_1(x))
    x = F.relu(self.FullyConnected_2(x))
    x = self.OutputLayer(x)

    return x


In [ ]:
torch.manual_seed(42)
model = MyModel().to(device)

# look at the architecture
print(model)
print(model.parameters)

In [ ]:
print(model.state_dict())

In [ ]:
# more advanced model summary - shows parameter count
# need to pass at least the size of input

import torchsummary

torchsummary.summary(model, (4,))

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
#training loop - simplified
epochs = 200
losses = []


for i in range(epochs):
  model.train()
  y_logits = model(X_train_torch)
  print(y_logits.dtype)
  y_pred = y_logits #   y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1) # explained later
  loss = loss_fn(y_pred, y_train_torch)
  # loss = loss_fn(y_pred, y_train_torch.unsqueeze(1)) # OR maybe like this?? Why?

  losses.append(loss.detach().cpu().numpy()) # https://pytorch.org/docs/stable/generated/torch.Tensor.detach.html


  if i % 10 == 0:
    print(f"Epoch: {i} | Loss: {loss}")
    # print(y_pred.shape)
    # print(y_train_torch.shape)


  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

In [ ]:
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")

In [ ]:
# simple evaluate
# https://pytorch.org/docs/stable/notes/autograd.html#locally-disable-grad-doc
predictions = []
model.eval() # some layers (like Dropout) have to be "notified" to change their behaviour in the inference mode
with torch.inference_mode(): # no backpropagation of errors, like torch.no_grad
  y_logits = model(X_test_torch) # OR model.forward(X_test_torch)
  y_pred = y_logits # y_pred = y_logits.argmax(dim=1)
  loss = loss_fn(y_pred, y_test_torch) #


  predicted = torch.abs(y_pred.round())  # Round predictions to the nearest integer
  correct = (predicted == y_test_torch).sum().item()
  accuracy = correct / len(y_test_torch)

  print(f"Loss: {loss}")
  print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
print(predicted)
print(y_test_torch)

In [ ]:
# more advanced architecture

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Set random seed for reproducibility
torch.manual_seed(42)

# Define the transformations
transform = transforms.Compose([
    transforms.RandomRotation(10),  # Random rotation of up to 10 degrees
    transforms.RandomHorizontalFlip(p=0.5),  # 50% chance of horizontal flip
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.5,), (0.5,))  # Normalize with mean 0.5 and std 0.5
])

# Load the FashionMNIST dataset
full_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

# Split the dataset into train and validation sets
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Load the test dataset
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Example of how to use the DataLoader
for batch_idx, (data, target) in enumerate(train_loader):
    print(f"Batch {batch_idx}")
    print(f"Data shape: {data.shape}")
    print(f"Target shape: {target.shape}")
    print(f"Sample target: {target[0]}")
    break

# Print dataset information
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.dataset.classes)}")
print(f"Class names: {train_dataset.dataset.classes}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Set random seed for reproducibility
torch.manual_seed(42)

# Define the MLP model
class FashionMNISTMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(FashionMNISTMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# Set up model parameters
input_size = 28 * 28  # FashionMNIST images are 28x28 pixels
hidden_size = 128
num_classes = 10  # FashionMNIST has 10 classes

# Create the model
model = FashionMNISTMLP(input_size, hidden_size, num_classes)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Print model summary
print(model)



# Calculate the number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")

In [ ]:
# check correctness and make sure the output has correct shape
sample_input = torch.randn(1, 1, 28, 28)  # (batch_size, channels, height, width)
sample_output = model(sample_input)
print(f"Sample input shape: {sample_input.shape}")
print(f"Sample output shape: {sample_output.shape}")

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt


model.to(device)

# Set random seed for reproducibility
torch.manual_seed(42)

# Hyperparameters
num_epochs = 2
learning_rate = 0.01

# Loss function and optimizer
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


In [ ]:
# Training loop
def train_model(model, train_loader, val_loader, loss_fn, optimizer, num_epochs):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs) ###

            # Convert labels to one-hot encoding
            targets = torch.zeros(outputs.shape).to(device)
            targets[torch.arange(targets.shape[0]), labels] = 1
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs) ###

                # Convert labels to one-hot encoding
                targets = torch.zeros(outputs.shape).to(device)
                targets[torch.arange(targets.shape[0]), labels] = 1

                loss = loss_fn(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted.cpu() == labels.cpu()).sum().item()

        val_loss /= len(val_loader.dataset)
        val_accuracy = 100 * correct / total
        val_losses.append(val_loss)

        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {train_loss:.4f}, '
              f'Val Loss: {val_loss:.4f}, '
              f'Val Accuracy: {val_accuracy:.2f}%')

    return train_losses, val_losses


In [ ]:
# Test loop
def test_model(model, test_loader):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs) ###

            # Convert labels to one-hot encoding
            targets = torch.zeros(outputs.shape).to(device)
            targets[torch.arange(targets.shape[0]), labels] = 1

            loss = loss_fn(outputs, targets)
            test_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted.cpu() == labels.cpu()).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100 * correct / total

    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')
    return test_loss, test_accuracy

In [ ]:
# Train the model
train_losses, val_losses = train_model(model, train_loader, val_loader, loss_fn, optimizer, num_epochs)
# OH no !!!


# Test the model
test_loss, test_accuracy = test_model(model, test_loader)

# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, num_epochs+1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
# CIFAR10

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Set random seed for reproducibility
torch.manual_seed(42)

# Define the transformations
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the CIFAR10 dataset
full_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)

# Split the dataset into train and validation sets
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Load the test dataset
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Example of how to use the DataLoader
for batch_idx, (data, target) in enumerate(train_loader):
    print(f"Batch {batch_idx}")
    print(f"Data shape: {data.shape}")
    print(f"Target shape: {target.shape}")
    print(f"Sample target: {target[0]}")
    break

# Print dataset information
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.dataset.classes)}")
print(f"Class names: {train_dataset.dataset.classes}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Set random seed for reproducibility
torch.manual_seed(42)

# Define the MLP model
class CIFAR10MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(CIFAR10MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# Set up model parameters
input_size = 3 * 32 * 32  # CIFAR10 images are 32x32 pixels with 3 color channels
hidden_size = 512  # Increased hidden size due to larger input
num_classes = 10  # CIFAR10 has 10 classes

# Create the model
model = CIFAR10MLP(input_size, hidden_size, num_classes)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Print model summary
print(model)

# Calculate the number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")


In [ ]:
# Example of how to use the model
sample_input = torch.randn(1, 3, 32, 32)  # (batch_size, channels, height, width)
sample_output = model(sample_input)
print(f"Sample input shape: {sample_input.shape}")
print(f"Sample output shape: {sample_output.shape}")

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Hyperparameters
num_epochs = 20
learning_rate = 0.01

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


In [ ]:
# Training loop
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)

            # Convert labels to one-hot encoding for MSE loss
            targets = torch.zeros(outputs.shape)
            targets[torch.arange(targets.shape[0]), labels] = 1

            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs)

                # Convert labels to one-hot encoding for MSE loss
                targets = torch.zeros(outputs.shape)
                targets[torch.arange(targets.shape[0]), labels] = 1

                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader.dataset)
        val_accuracy = 100 * correct / total
        val_losses.append(val_loss)

        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {train_loss:.4f}, '
              f'Val Loss: {val_loss:.4f}, '
              f'Val Accuracy: {val_accuracy:.2f}%')

    return train_losses, val_losses



In [ ]:
# Test loop
def test_model(model, test_loader):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)

            # Convert labels to one-hot encoding for MSE loss
            targets = torch.zeros(outputs.shape)
            targets[torch.arange(targets.shape[0]), labels] = 1

            loss = criterion(outputs, targets)
            test_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted.cpu() == labels).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100 * correct / total

    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')
    return test_loss, test_accuracy



In [ ]:
# can this model be faster?

# Train the model
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs)

# Test the model
test_loss, test_accuracy = test_model(model, test_loader)

# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, num_epochs+1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()